# Chuẩn bị các thư viện cần thiết

In [2]:
# Install selenium
!pip install -qq selenium

# Tạo thư mục để chứa data
!mkdir data

A subdirectory or file data already exists.


In [3]:
# selenium import
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException

# other imports
import os
import json
from urllib.parse import urljoin

In [4]:
chrome_options = webdriver.ChromeOptions()

chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-gpu")

# Thu thập dữ liệu

In [5]:
driver = webdriver.Chrome(options=chrome_options)

In [20]:
URL = "https://undergrad.tdtu.edu.vn/hoc-vu/quy-che-chuc-va-quan-ly-dao-tao-trinh-do-dai-hoc-khoa-ts2021-tro-ve-sau"
driver.get(URL)

In [21]:
data = []

In [22]:
chapters = driver.find_elements(by=By.CLASS_NAME, value="ckeditor-accordion-container")

for chap in chapters:
    titles = chap.find_elements(by=By.TAG_NAME, value="dt")
    contents = chap.find_elements(by=By.TAG_NAME, value="dd")
    for title, content in zip(titles, contents):
        article = {}
        article['title'] = title.find_element(by=By.TAG_NAME, value='a').text
        # print(article['title'])
        # title.click()
        doc = content.find_elements(by=By.CSS_SELECTOR, value='p.text-align-justify')
        article['content'] = []
        for p in doc:
            text = driver.execute_script("return arguments[0].textContent;", p)
            article['content'].append(text)
        article['content'] = '\n'.join(article['content'])
        data.append(article)

In [23]:
print(len(data))

37


In [24]:
URLS = [
    'https://www.tdtu.edu.vn/gioi-thieu/lich-su-hinh-thanh-va-muc-tieu',
    'https://www.tdtu.edu.vn/gioi-thieu/triet-ly-giao-duc-su-mang-tam-nhin-chinh-sach-chat-luong',
    'https://www.tdtu.edu.vn/gioi-thieu/dinh-huong-phat-trien',
]

In [25]:
for url in URLS:
    driver.get(url)
    title = driver.find_element(by=By.CLASS_NAME, value='page-title').text
    content = driver.find_element(by=By.TAG_NAME, value='article')
    paragraphs = content.find_elements(by=By.TAG_NAME, value='p')
    article = {}
    article['title'] = title
    article['content'] = []
    for p in paragraphs:
        if p.text != '':
            article['content'].append(p.text)
    article['content'] = '\n'.join(article['content'])
    data.append(article)

# Xử lý title

In [26]:
for i in data[:5]:
    print(i['title'])

Điều 1. Phạm vi và đối tượng áp dụng
Điều 2. Chương trình đào tạo
Điều 3. Phương thức tổ chức đào tạo và hình thức đào tạo
Điều 4. Tín chỉ, Môn học trong Chương trình đào tạo
Điều 5. Thời gian đào tạo


In [28]:
new_data = []
for i in data:
    a = i['title'].split()
    if a[0] == 'Điều':
        a = a[2:]
    i['title'] = ' '.join(a)
    new_data.append(i)

In [29]:
data = new_data
print(data[:2])

[{'title': 'Phạm vi và đối tượng áp dụng', 'content': '1. Quy chế này quy định tổ chức và quản lý đào tạo trình độ đại học hệ chính quy theo hệ thống tín chỉ tại Trường Đại học Tôn Đức Thắng, bao gồm: Chương trình đào tạo, tổ chức đào tạo, đánh giá kết quả học tập, xét và công nhận tốt nghiệp và những quy định khác.\n2. Quy chế này áp dụng đối với sinh viên chương trình đào tạo tiêu chuẩn, chương trình đào tạo chất lượng cao, chương trình đào tạo giảng dạy bằng tiếng Anh hệ chính quy trình độ đại học tại Trường Đại học Tôn Đức Thắng (sau đây gọi tắt là Trường) theo hình thức tích lũy tín chỉ.'}, {'title': 'Chương trình đào tạo', 'content': '1. Chương trình đào tạo (CTĐT) được xây dựng theo đơn vị tín chỉ, cấu trúc từ các môn học hoặc học phần (gọi chung là môn học), trong đó phải có đủ các môn học bắt buộc và đáp ứng chuẩn chương trình đào tạo theo quy định hiện hành của Bộ Giáo dục và Đào tạo (GD&ĐT). Trong trường hợp đào tạo song ngành hoặc ngành chính - ngành phụ, chương trình đào t

# Lưu dữ liệu vào cơ sở dữ liệu

In [31]:
import os
import json
import re
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from pyvi import ViTokenizer
from tqdm.notebook import tqdm
import pandas as pd

## Kết nối cơ sở dữ liệu

In [33]:
import mysql.connector

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password=""
)

mycursor = mydb.cursor()

mycursor.execute("SHOW DATABASES")

for x in mycursor:
  print(x)

('information_schema',)
('mydatabase',)
('mysql',)
('news',)
('online_marketing',)
('performance_schema',)
('phpmyadmin',)
('qlvideo',)
('shop',)


In [34]:
DATABASE_NAME = "quy_dinh_tdtu"

In [35]:
mycursor.execute(f"CREATE DATABASE {DATABASE_NAME}")

In [36]:
import mysql.connector

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="",
  database=DATABASE_NAME
)

print(mydb)

In [37]:
cursor = mydb.cursor()

query = """
    create table thong_tin_quy_dinh (
        id INT AUTO_INCREMENT PRIMARY KEY,
        title VARCHAR(255),
        content TEXT
    )
"""

cursor.execute(query)

In [39]:
query = "INSERT INTO thong_tin_quy_dinh (title, content) VALUES (%s, %s)"

for doc in data:
    title = doc['title']
    content = doc['content']
    values = (title, content)
    cursor.execute(query, values)

mydb.commit()

In [40]:
cursor.close()
mydb.close()